# PLINK / GWAS QC Workflow Agent
## Colab 05 — Final Report + Gradio Web App

這份 Notebook 把前面 01～04 的核心功能整合成一個可以直接操作的期末專案成品。

完整流程：

```text
User Study Information
↓
RAG Retrieval
↓
Retrieval Coverage Check
↓
Groq QC Planning
↓
Python Plan Validator
↓
FAIL → Groq Revise → Recheck
↓ PASS
Groq Command Generator
↓
Python Command Validator
↓
FAIL → Groq Revise → Recheck
↓ PASS
Final Report
↓
Gradio UI
```

### 最終 UI 會顯示
- Retrieved Knowledge
- QC Plan
- Plan Check
- PLINK Commands
- Command Check
- Final Report

## 0. 使用前確認

Colab 左側 **Secrets** 中需有：

```text
Groq
```

並開啟「筆記本存取權」。

此 Notebook 會從你的 GitHub Repository 下載 Wiki：

```text
b156698-commits/PLINK-GWAS-Quality-Control-Wiki
```

## 1. 安裝套件

In [ ]:
!pip -q install groq sentence-transformers faiss-cpu gradio

## 2. Imports + 設定

In [ ]:
import re
import json
import shlex
import shutil
from pathlib import Path
from copy import deepcopy

import numpy as np
import faiss
import gradio as gr

from sentence_transformers import SentenceTransformer
from groq import Groq, BadRequestError, RateLimitError
from google.colab import userdata

GROQ_API_KEY = userdata.get("Groq")

if not GROQ_API_KEY:
    raise ValueError("找不到 Colab Secret：Groq")

client = Groq(api_key=GROQ_API_KEY)

MODEL_NAME = "openai/gpt-oss-20b"
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

TOP_K_PER_QUERY = 3
MAX_PLAN_ITERATIONS = 3
MAX_COMMAND_ITERATIONS = 3
SEED = 123

print("Environment: READY")

# Part A — Wiki / RAG

## 3. Clone GitHub Wiki

In [ ]:
REPO_URL = "https://github.com/b156698-commits/PLINK-GWAS-Quality-Control-Wiki.git"
REPO_DIR = Path("/content/PLINK-GWAS-Quality-Control-Wiki")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

!git clone -q {REPO_URL} {REPO_DIR}

wiki_dir = REPO_DIR / "wiki"
wiki_files = sorted(wiki_dir.glob("*.md"))

print("Wiki files:", len(wiki_files))

for f in wiki_files:
    print("-", f.name)

## 4. Read + Chunk + Embedding + FAISS

In [ ]:
def extract_source_ids(text):
    return sorted(
        set(
            re.findall(
                r"\bS\d{2}\b",
                text
            )
        )
    )


def chunk_markdown(
    text,
    max_chars=1200
):
    paragraphs = [
        p.strip()
        for p in re.split(
            r"\n\s*\n",
            text
        )
        if p.strip()
    ]

    chunks = []
    current = ""

    for p in paragraphs:
        candidate = (
            p
            if not current
            else current + "\n\n" + p
        )

        if len(candidate) <= max_chars:
            current = candidate
        else:
            if current:
                chunks.append(current)
            current = p

    if current:
        chunks.append(current)

    return chunks


documents = []

for file_path in wiki_files:
    text = file_path.read_text(
        encoding="utf-8"
    )

    documents.append({
        "source_file": file_path.name,
        "source_ids": extract_source_ids(text),
        "text": text
    })


chunks = []

for doc in documents:
    for i, chunk_text in enumerate(
        chunk_markdown(
            doc["text"]
        )
    ):
        chunks.append({
            "chunk_id":
                f'{doc["source_file"]}::chunk_{i+1}',
            "source_file":
                doc["source_file"],
            "source_ids":
                doc["source_ids"],
            "text":
                chunk_text
        })


embedding_model = SentenceTransformer(
    EMBEDDING_MODEL
)

texts = [
    c["text"]
    for c in chunks
]

embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

embeddings = np.asarray(
    embeddings,
    dtype="float32"
)

index = faiss.IndexFlatIP(
    embeddings.shape[1]
)

index.add(embeddings)

print("Documents:", len(documents))
print("Chunks:", len(chunks))
print("FAISS index:", index.ntotal)

## 5. Retrieval Functions

In [ ]:
def retrieve(
    query,
    top_k=TOP_K_PER_QUERY
):
    query_vector = embedding_model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(
        query_vector,
        top_k
    )

    results = []

    for score, idx in zip(
        scores[0],
        indices[0]
    ):
        item = chunks[
            int(idx)
        ].copy()

        item["score"] = float(score)

        results.append(item)

    return results


def deduplicate_results(
    results
):
    best = {}

    for r in results:
        key = r["chunk_id"]

        if (
            key not in best
            or r["score"]
            > best[key]["score"]
        ):
            best[key] = r

    return sorted(
        best.values(),
        key=lambda x: x["score"],
        reverse=True
    )


def build_rag_context(
    results
):
    blocks = []

    for i, r in enumerate(
        results,
        start=1
    ):
        ids = (
            ", ".join(
                r["source_ids"]
            )
            if r["source_ids"]
            else "N/A"
        )

        blocks.append(
            f"[Retrieved Chunk {i}]\n"
            f"Source file: "
            f"{r['source_file']}\n"
            f"Source IDs: {ids}\n\n"
            f"{r['text']}"
        )

    return "\n\n".join(
        blocks
    )

## 6. Study-specific Retrieval

In [ ]:
def build_required_steps(
    study_info
):
    required = [
        "Load PLINK Binary Files",
        "Variant Missingness",
        "Sample Missingness",
        "MAF Filtering",
        "HWE Filtering"
    ]

    # LD pruning is needed here only when we need a relatively
    # LD-independent SNP subset for Relatedness and/or PCA.
    if (
        study_info.get("need_relatedness")
        or study_info.get("need_pca")
    ):
        required.append(
            "LD Pruning"
        )

    if study_info.get(
        "need_relatedness"
    ):
        required.append(
            "Relatedness"
        )

    if study_info.get(
        "need_pca"
    ):
        required.append(
            "PCA"
        )

    required.append(
        "Association Analysis"
    )

    return required


def required_wiki_files(
    study_info
):
    files = {
        "01-plink-basics.md",
        "02-missingness.md",
        "03-maf-hwe.md",
        "07-association.md"
    }

    if (
        study_info.get("need_relatedness")
        or study_info.get("need_pca")
    ):
        files.add(
            "04-ld-pruning.md"
        )

    if study_info.get(
        "need_relatedness"
    ):
        files.add(
            "05-relatedness.md"
        )

    if study_info.get(
        "need_pca"
    ):
        files.add(
            "06-pca.md"
        )

    return files


def retrieve_for_study(
    study_info
):
    topic_queries = [
        "PLINK binary files bed bim fam bfile",
        "PLINK variant missingness --geno "
        "and sample missingness --mind",
        "PLINK MAF filtering and HWE filtering"
    ]

    if (
        study_info.get("need_relatedness")
        or study_info.get("need_pca")
    ):
        topic_queries.append(
            "PLINK LD pruning --indep-pairwise"
        )

    if study_info.get(
        "need_relatedness"
    ):
        topic_queries.append(
            "PLINK relatedness IBD PI_HAT "
            "--genome and LD pruning"
        )

    if study_info.get(
        "need_pca"
    ):
        topic_queries.append(
            "PLINK PCA population structure "
            "eigenvec eigenval"
        )

    topic_queries.append(
        "PLINK case-control logistic association "
        "phenotype covariate pheno-name covar-name"
    )

    all_results = []

    for q in topic_queries:
        all_results.extend(
            retrieve(
                q,
                top_k=TOP_K_PER_QUERY
            )
        )

    results = deduplicate_results(
        all_results
    )

    required_files = required_wiki_files(
        study_info
    )

    present_files = {
        r["source_file"]
        for r in results
    }

    missing_files = (
        required_files
        - present_files
    )

    result_map = {
        r["chunk_id"]: r
        for r in results
    }

    for c in chunks:
        if (
            c["source_file"]
            in missing_files
        ):
            item = c.copy()
            item["score"] = 0.0
            result_map[
                item["chunk_id"]
            ] = item

    results = list(
        result_map.values()
    )

    results.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    allowed_source_ids = sorted({
        sid
        for r in results
        for sid in r["source_ids"]
    })

    return (
        results,
        allowed_source_ids
    )

# Part B — Planning Agent

## 7. QC Plan JSON Schema

In [ ]:
QC_PLAN_SCHEMA = {
    "type": "object",
    "properties": {
        "qc_plan": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "step": {
                        "type": "integer"
                    },
                    "name": {
                        "type": "string"
                    },
                    "purpose": {
                        "type": "string"
                    },
                    "plink_flag": {
                        "type": "string"
                    },
                    "example_setting": {
                        "type": [
                            "string",
                            "null"
                        ]
                    },
                    "note": {
                        "type": "string"
                    },
                    "source": {
                        "type": "string"
                    }
                },
                "required": [
                    "step",
                    "name",
                    "purpose",
                    "plink_flag",
                    "example_setting",
                    "note",
                    "source"
                ],
                "additionalProperties": False
            }
        }
    },
    "required": [
        "qc_plan"
    ],
    "additionalProperties": False
}

## 8. Planner Prompt

In [ ]:
PLANNER_SYSTEM_PROMPT = '''
You are the planning component of a PLINK / GWAS QC Workflow Agent.

Generate a QC plan using only the retrieved Wiki context.

STRICT RULES:
- Include EVERY step listed in REQUIRED STEP CHECKLIST.
- Every step MUST use one or more valid Source IDs from ALLOWED SOURCE IDS.
- The source field MUST contain only IDs such as S02 or S05, S06.
- NEVER write "尚待驗證", "unknown", "N/A", or prose in the source field.
- Do not invent PLINK flags.
- Threshold values are examples, not universal standards.
- LD pruning must precede relatedness/PCA when requested.
- Do not claim the final GWAS must use only LD-pruned SNPs.
- PCA must precede association when requested.
- Return only the requested structured output.
'''.strip()

## 9. Plan JSON Helper

In [ ]:
REQUIRED_STEP_KEYS = {
    "step",
    "name",
    "purpose",
    "plink_flag",
    "example_setting",
    "note",
    "source"
}


def validate_plan_json_shape(data):
    errors = []

    if not isinstance(data, dict):
        return ["Top-level response must be a JSON object."]

    if "qc_plan" not in data:
        return ["Missing top-level key: qc_plan."]

    plan = data["qc_plan"]

    if not isinstance(plan, list) or len(plan) == 0:
        return ["qc_plan must be a non-empty list."]

    for i, step in enumerate(plan, start=1):
        if not isinstance(step, dict):
            errors.append(f"Step {i} must be a JSON object.")
            continue

        missing = REQUIRED_STEP_KEYS - set(step.keys())

        if missing:
            errors.append(
                f"Step {i} missing keys: {sorted(missing)}"
            )

        if "step" in step and not isinstance(step["step"], int):
            errors.append(f"Step {i}: step must be integer.")

        if "name" in step and not isinstance(step["name"], str):
            errors.append(f"Step {i}: name must be string.")

        if "source" in step and not isinstance(step["source"], str):
            errors.append(f"Step {i}: source must be string.")

    return errors


def call_plan_json(
    prompt,
    schema_name=None,
    max_attempts=3
):
    format_prompt = (
        prompt
        + "\n\nOUTPUT FORMAT REQUIREMENTS:\n"
          "- Return ONLY one valid JSON object.\n"
          "- Do not use Markdown code fences.\n"
          "- Top-level object must contain exactly one key: qc_plan.\n"
          "- qc_plan must be a non-empty array.\n"
          "- Every step must contain these keys:\n"
          "  step, name, purpose, plink_flag, example_setting, note, source.\n"
          "- source must contain only valid Source IDs such as S02 or S05, S06.\n"
    )

    last_error = None

    for attempt in range(1, max_attempts + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {
                        "role": "system",
                        "content": PLANNER_SYSTEM_PROMPT
                    },
                    {
                        "role": "user",
                        "content": format_prompt
                    }
                ],
                temperature=0.0,
                seed=SEED,
                reasoning_effort="low",
                max_completion_tokens=8192,
                response_format={
                    "type": "json_object"
                }
            )

            raw = response.choices[0].message.content
            data = json.loads(raw)

            shape_errors = validate_plan_json_shape(data)

            if not shape_errors:
                return data

            last_error = (
                "Local JSON shape validation failed: "
                + " | ".join(shape_errors)
            )

            format_prompt += (
                "\n\nPREVIOUS OUTPUT FAILED THESE LOCAL CHECKS:\n"
                + "\n".join(f"- {e}" for e in shape_errors)
                + "\nReturn the COMPLETE corrected JSON object."
            )

        except RateLimitError:
            # Rate limit is not a malformed-output problem.
            # Retrying immediately only wastes time, so stop at once.
            raise

        except Exception as e:
            last_error = f"{type(e).__name__}: {e}"

    raise RuntimeError(
        "Planner could not produce usable JSON after "
        f"{max_attempts} attempts. Last error: {last_error}"
    )

## 10. Generate / Revise QC Plan

In [ ]:
def generate_qc_plan(
    study_info,
    required_steps,
    rag_context,
    allowed_source_ids
):
    prompt = (
        "STUDY INFORMATION\n"
        + json.dumps(
            study_info,
            ensure_ascii=False,
            indent=2
        )
        + "\n\nREQUIRED STEP CHECKLIST\n"
        + json.dumps(
            required_steps,
            ensure_ascii=False,
            indent=2
        )
        + "\n\nALLOWED SOURCE IDS\n"
        + json.dumps(
            allowed_source_ids,
            ensure_ascii=False
        )
        + "\n\nRETRIEVED WIKI CONTEXT\n"
        + rag_context
        + "\n\nGenerate the complete QC plan."
    )

    data = call_plan_json(
        prompt,
        "gwas_qc_plan"
    )

    return data["qc_plan"]


def revise_qc_plan(
    study_info,
    required_steps,
    rag_context,
    allowed_source_ids,
    current_plan,
    errors
):
    prompt = (
        "STUDY INFORMATION\n"
        + json.dumps(
            study_info,
            ensure_ascii=False,
            indent=2
        )
        + "\n\nREQUIRED STEP CHECKLIST\n"
        + json.dumps(
            required_steps,
            ensure_ascii=False,
            indent=2
        )
        + "\n\nALLOWED SOURCE IDS\n"
        + json.dumps(
            allowed_source_ids,
            ensure_ascii=False
        )
        + "\n\nCURRENT PLAN\n"
        + json.dumps(
            current_plan,
            ensure_ascii=False,
            indent=2
        )
        + "\n\nVALIDATOR ERRORS\n"
        + json.dumps(
            errors,
            ensure_ascii=False,
            indent=2
        )
        + "\n\nPatch the plan minimally. "
          "Fix all errors. "
          "Do not remove unrelated valid steps. "
          "Return full corrected plan."
    )

    data = call_plan_json(
        prompt,
        "revised_gwas_qc_plan"
    )

    return data["qc_plan"]

## 11. Plan Validator

In [ ]:
def normalize_name(
    name
):
    return str(
        name
    ).strip().lower()


def find_step_index(
    plan,
    keyword
):
    keyword = keyword.lower()

    for i, step in enumerate(
        plan
    ):
        if keyword in normalize_name(
            step.get(
                "name",
                ""
            )
        ):
            return i

    return None


def parse_source_ids(
    source_text
):
    return [
        x.strip()
        for x in str(
            source_text
        ).split(",")
        if x.strip()
    ]


def has_required_step(
    plan_names,
    required_name
):
    req = required_name.lower()

    aliases = {
        "load plink binary files":
            [
                "load plink",
                "plink binary",
                "input"
            ],
        "variant missingness":
            ["variant missingness"],
        "sample missingness":
            ["sample missingness"],
        "maf filtering":
            ["maf"],
        "hwe filtering":
            ["hwe"],
        "ld pruning":
            ["ld pruning"],
        "relatedness":
            ["relatedness"],
        "pca":
            ["pca"],
        "association analysis":
            ["association"]
    }

    patterns = aliases.get(
        req,
        [req]
    )

    return any(
        any(
            p in n
            for p in patterns
        )
        for n in plan_names
    )


def validate_qc_plan(
    study_info,
    required_steps,
    plan,
    allowed_source_ids
):
    errors = []

    names = [
        normalize_name(
            step.get(
                "name",
                ""
            )
        )
        for step in plan
    ]

    for required in required_steps:
        if not has_required_step(
            names,
            required
        ):
            errors.append(
                f"Missing required step: "
                f"{required}."
            )

    ld_idx = find_step_index(
        plan,
        "ld pruning"
    )

    rel_idx = find_step_index(
        plan,
        "relatedness"
    )

    pca_idx = find_step_index(
        plan,
        "pca"
    )

    assoc_idx = find_step_index(
        plan,
        "association"
    )

    if (
        study_info.get(
            "need_relatedness"
        )
        and rel_idx is not None
    ):
        if ld_idx is None:
            errors.append(
                "Relatedness requires "
                "LD Pruning."
            )
        elif ld_idx > rel_idx:
            errors.append(
                "LD Pruning should occur "
                "before Relatedness."
            )

    if (
        study_info.get(
            "need_pca"
        )
        and pca_idx is not None
    ):
        if ld_idx is None:
            errors.append(
                "PCA requires LD Pruning."
            )
        elif ld_idx > pca_idx:
            errors.append(
                "LD Pruning should occur "
                "before PCA."
            )

        if (
            assoc_idx is not None
            and pca_idx > assoc_idx
        ):
            errors.append(
                "PCA should occur before "
                "Association Analysis."
            )

    allowed = set(
        allowed_source_ids
    )

    for i, step in enumerate(
        plan,
        start=1
    ):
        source_ids = (
            parse_source_ids(
                step.get(
                    "source",
                    ""
                )
            )
        )

        if not source_ids:
            errors.append(
                f"Step {i} has no source."
            )
            continue

        for sid in source_ids:
            if not re.fullmatch(
                r"S\d{2}",
                sid
            ):
                errors.append(
                    f"Step {i} has invalid "
                    f"source: {sid}"
                )
            elif sid not in allowed:
                errors.append(
                    f"Step {i} uses source "
                    f"outside retrieved "
                    f"context: {sid}"
                )

    bad_phrases = [
        "standard threshold",
        "universal threshold",
        "must use",
        "fixed threshold"
    ]

    for i, step in enumerate(
        plan,
        start=1
    ):
        combined = " ".join([
            str(
                step.get(
                    "example_setting",
                    ""
                )
            ),
            str(
                step.get(
                    "note",
                    ""
                )
            ),
            str(
                step.get(
                    "purpose",
                    ""
                )
            )
        ]).lower()

        for phrase in bad_phrases:
            if phrase in combined:
                errors.append(
                    f"Step {i} may describe "
                    "an example threshold "
                    "as universal."
                )

    for i, step in enumerate(
        plan,
        start=1
    ):
        if "association" in normalize_name(
            step.get(
                "name",
                ""
            )
        ):
            combined = " ".join([
                str(
                    step.get(
                        "purpose",
                        ""
                    )
                ),
                str(
                    step.get(
                        "note",
                        ""
                    )
                )
            ]).lower()

            risky = [
                "only ld-pruned",
                "only use ld-pruned",
                "final gwas only",
                "only the ld-pruned"
            ]

            if any(
                x in combined
                for x in risky
            ):
                errors.append(
                    "Association Analysis "
                    "may incorrectly use only "
                    "LD-pruned SNPs."
                )

    return errors

# Part C — Command Generator

## 12. Command Schema + Prompt

In [ ]:
COMMAND_SCHEMA = {
    "type": "object",
    "properties": {
        "command_plan": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "step": {
                        "type": "integer"
                    },
                    "step_name": {
                        "type": "string"
                    },
                    "commands": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        }
                    },
                    "purpose": {
                        "type": "string"
                    },
                    "source": {
                        "type": "string"
                    }
                },
                "required": [
                    "step",
                    "step_name",
                    "commands",
                    "purpose",
                    "source"
                ],
                "additionalProperties": False
            }
        }
    },
    "required": [
        "command_plan"
    ],
    "additionalProperties": False
}


COMMAND_SYSTEM_PROMPT = '''
You are the PLINK command-generation component of a GWAS QC Workflow Agent.

STRICT RULES:
- Follow the VERIFIED QC PLAN.
- Do not invent PLINK flags.
- Every command begins with plink.
- Every command includes --bfile and --out.
- Use --make-bed when filtering should create a new binary dataset.
- LD-pruned datasets may be used for Relatedness/PCA.
- Final association must use the post-QC association dataset, not only LD-pruned SNPs.
- Case-control association uses --logistic.
- --pheno-name requires --pheno.
- --covar-name requires --covar.
- Include study covariates when supplied.
- Preserve Source IDs from the verified QC plan.
- Return only structured JSON.
'''.strip()

## 13. Robust Command JSON Helper

In [ ]:
REQUIRED_COMMAND_KEYS = {
    "step",
    "step_name",
    "commands",
    "purpose",
    "source"
}


def validate_command_json_shape(data):
    errors = []

    if not isinstance(data, dict):
        return ["Top-level output must be a JSON object."]

    if "command_plan" not in data:
        return ["Missing top-level key: command_plan."]

    plan = data["command_plan"]

    if not isinstance(plan, list) or len(plan) == 0:
        return ["command_plan must be a non-empty list."]

    for i, item in enumerate(plan, start=1):
        if not isinstance(item, dict):
            errors.append(f"Item {i} must be a JSON object.")
            continue

        missing = REQUIRED_COMMAND_KEYS - set(item.keys())

        if missing:
            errors.append(
                f"Item {i} missing keys: {sorted(missing)}"
            )

        commands = item.get("commands")

        if (
            "commands" in item
            and (
                not isinstance(commands, list)
                or len(commands) == 0
                or not all(isinstance(x, str) for x in commands)
            )
        ):
            errors.append(
                f"Item {i}: commands must be a non-empty list of strings."
            )

    return errors


def call_command_json(
    prompt,
    schema_name=None,
    max_attempts=3
):
    format_prompt = (
        prompt
        + "\n\nOUTPUT FORMAT REQUIREMENTS:\n"
          "- Return ONLY one valid JSON object.\n"
          "- Do not use Markdown code fences.\n"
          "- Top-level object must contain exactly one key: command_plan.\n"
          "- command_plan must be a non-empty array.\n"
          "- Every item must contain these keys:\n"
          "  step, step_name, commands, purpose, source.\n"
          "- commands must be a non-empty array of strings.\n"
    )

    last_error = None

    for attempt in range(1, max_attempts + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {
                        "role": "system",
                        "content": COMMAND_SYSTEM_PROMPT
                    },
                    {
                        "role": "user",
                        "content": format_prompt
                    }
                ],
                temperature=0.0,
                seed=SEED,
                reasoning_effort="low",
                max_completion_tokens=8192,
                response_format={
                    "type": "json_object"
                }
            )

            raw = response.choices[0].message.content
            data = json.loads(raw)

            shape_errors = validate_command_json_shape(data)

            if not shape_errors:
                return data

            last_error = (
                "Local JSON shape validation failed: "
                + " | ".join(shape_errors)
            )

            format_prompt += (
                "\n\nPREVIOUS OUTPUT FAILED THESE LOCAL CHECKS:\n"
                + "\n".join(f"- {e}" for e in shape_errors)
                + "\nReturn the COMPLETE corrected JSON object."
            )

        except RateLimitError:
            # Rate limit is not a malformed-output problem.
            # Retrying immediately only wastes time, so stop at once.
            raise

        except Exception as e:
            last_error = f"{type(e).__name__}: {e}"

    raise RuntimeError(
        "Command generator could not produce usable JSON after "
        f"{max_attempts} attempts. Last error: {last_error}"
    )

## 14. Generate / Revise Commands

In [ ]:
def generate_commands(
    study_info,
    verified_qc_plan
):
    prompt = (
        "STUDY INFORMATION\n"
        + json.dumps(
            study_info,
            ensure_ascii=False,
            indent=2
        )
        + "\n\nVERIFIED QC PLAN\n"
        + json.dumps(
            verified_qc_plan,
            ensure_ascii=False,
            indent=2
        )
        + "\n\nGenerate PLINK commands."
    )

    data = call_command_json(
        prompt,
        "plink_command_plan"
    )

    return data[
        "command_plan"
    ]


def revise_commands(
    study_info,
    verified_qc_plan,
    current_command_plan,
    errors
):
    prompt = (
        "STUDY INFORMATION\n"
        + json.dumps(
            study_info,
            ensure_ascii=False,
            indent=2
        )
        + "\n\nVERIFIED QC PLAN\n"
        + json.dumps(
            verified_qc_plan,
            ensure_ascii=False,
            indent=2
        )
        + "\n\nCURRENT COMMAND PLAN\n"
        + json.dumps(
            current_command_plan,
            ensure_ascii=False,
            indent=2
        )
        + "\n\nCOMMAND VALIDATOR ERRORS\n"
        + json.dumps(
            errors,
            ensure_ascii=False,
            indent=2
        )
        + "\n\nPatch minimally. "
          "Fix every validator error. "
          "Do not remove unrelated valid "
          "commands. Return full plan."
    )

    data = call_command_json(
        prompt,
        "revised_plink_command_plan"
    )

    return data[
        "command_plan"
    ]

## 15. Command Validator

In [ ]:
ALLOWED_FLAGS = {
    "--bfile",
    "--geno",
    "--mind",
    "--maf",
    "--hwe",
    "--indep-pairwise",
    "--extract",
    "--genome",
    "--pca",
    "--pheno",
    "--pheno-name",
    "--covar",
    "--covar-name",
    "--logistic",
    "--1",
    "--make-bed",
    "--out"
}


def normalize_command(
    command
):
    command = command.replace(
        "\\\n",
        " "
    )

    return re.sub(
        r"\s+",
        " ",
        command
    ).strip()


def tokenize_command(
    command
):
    try:
        return shlex.split(
            normalize_command(
                command
            )
        )
    except Exception:
        return normalize_command(
            command
        ).split()


def get_flags(
    command
):
    return [
        t
        for t in tokenize_command(
            command
        )
        if t.startswith("--")
    ]


def get_flag_value(
    command,
    flag
):
    tokens = tokenize_command(
        command
    )

    if flag not in tokens:
        return None

    idx = tokens.index(
        flag
    )

    if idx + 1 >= len(
        tokens
    ):
        return None

    nxt = tokens[
        idx + 1
    ]

    if nxt.startswith("--"):
        return None

    return nxt


def validate_command_plan(
    study_info,
    verified_qc_plan,
    command_plan
):
    errors = []

    expected_step_flags = {
        "load plink binary files":
            "--bfile",
        "variant missingness":
            "--geno",
        "sample missingness":
            "--mind",
        "maf filtering":
            "--maf",
        "hwe filtering":
            "--hwe",
        "ld pruning":
            "--indep-pairwise",
        "relatedness":
            "--genome",
        "pca":
            "--pca",
        "association analysis":
            "--logistic"
    }

    plan_source_map = {
        str(
            x.get(
                "name",
                ""
            )
        ).strip().lower():
        str(
            x.get(
                "source",
                ""
            )
        ).strip()
        for x in verified_qc_plan
    }

    for item_idx, item in enumerate(
        command_plan,
        start=1
    ):
        step_name = str(
            item.get(
                "step_name",
                ""
            )
        ).strip()

        step_key = (
            step_name.lower()
        )

        commands = item.get(
            "commands",
            []
        )

        item_source = str(
            item.get(
                "source",
                ""
            )
        ).strip()

        matching_source = None

        for plan_name, source in (
            plan_source_map.items()
        ):
            if (
                step_key in plan_name
                or plan_name in step_key
            ):
                matching_source = source
                break

        if (
            matching_source
            and item_source
            != matching_source
        ):
            errors.append(
                f"{step_name}: source "
                "does not match verified "
                "plan."
            )

        all_flags = []

        for cmd_idx, command in enumerate(
            commands,
            start=1
        ):
            tokens = tokenize_command(
                command
            )

            if (
                not tokens
                or tokens[0].lower()
                != "plink"
            ):
                errors.append(
                    f"{step_name}, command "
                    f"{cmd_idx}: must begin "
                    "with plink."
                )
                continue

            flags = get_flags(
                command
            )

            all_flags.extend(
                flags
            )

            for flag in flags:
                if (
                    flag
                    not in ALLOWED_FLAGS
                ):
                    errors.append(
                        f"{step_name}: "
                        f"Unknown PLINK flag "
                        f"{flag}."
                    )

            if "--bfile" not in flags:
                errors.append(
                    f"{step_name}: "
                    "Missing --bfile."
                )

            if "--out" not in flags:
                errors.append(
                    f"{step_name}: "
                    "Missing --out."
                )

            if (
                "--pheno-name"
                in flags
                and "--pheno"
                not in flags
            ):
                errors.append(
                    f"{step_name}: "
                    "--pheno-name requires "
                    "--pheno."
                )

            if (
                "--covar-name"
                in flags
                and "--covar"
                not in flags
            ):
                errors.append(
                    f"{step_name}: "
                    "--covar-name requires "
                    "--covar."
                )

        expected = None

        for key, flag in (
            expected_step_flags.items()
        ):
            if key in step_key:
                expected = flag
                break

        if (
            expected
            and expected
            not in all_flags
        ):
            errors.append(
                f"{step_name}: missing "
                f"expected flag "
                f"{expected}."
            )

        if (
            "association"
            in step_key
        ):
            combined_flags = set()

            for cmd in commands:
                combined_flags.update(
                    get_flags(
                        cmd
                    )
                )

                bfile = get_flag_value(
                    cmd,
                    "--bfile"
                )

                if (
                    bfile
                    and (
                        "prune"
                        in bfile.lower()
                        or "ld_"
                        in bfile.lower()
                    )
                ):
                    errors.append(
                        "Association Analysis "
                        "appears to use an "
                        "LD-pruned final "
                        "GWAS dataset."
                    )

            if (
                study_info.get(
                    "study_design"
                )
                == "case-control"
                and study_info.get(
                    "association"
                )
                == "logistic"
                and "--logistic"
                not in combined_flags
            ):
                errors.append(
                    "Association Analysis "
                    "requires --logistic."
                )

            if (
                study_info.get(
                    "covariates"
                )
                and "--covar"
                not in combined_flags
            ):
                errors.append(
                    "Association Analysis "
                    "has covariates but "
                    "no --covar."
                )

    return errors

# Part D — Formatting

## 16. Markdown Formatters

In [ ]:
def format_retrieved_knowledge(
    retrieved_docs
):
    by_file = {}

    for r in retrieved_docs:
        file = r["source_file"]

        if file not in by_file:
            by_file[file] = {
                "ids":
                    r["source_ids"],
                "score":
                    r["score"]
            }
        else:
            by_file[file]["score"] = max(
                by_file[file]["score"],
                r["score"]
            )

    lines = [
        "## Retrieved Wiki Knowledge",
        ""
    ]

    for file, info in sorted(
        by_file.items()
    ):
        ids = (
            ", ".join(
                info["ids"]
            )
            if info["ids"]
            else "N/A"
        )

        lines.append(
            f"- **{file}** "
            f"｜ Source: {ids} "
            f"｜ similarity: "
            f"{info['score']:.3f}"
        )

    return "\n".join(
        lines
    )


def format_qc_plan(
    plan
):
    lines = [
        "## Recommended QC Plan",
        ""
    ]

    for step in plan:
        lines.extend([
            f"### {step['step']}. "
            f"{step['name']}",
            f"- **Purpose:** "
            f"{step['purpose']}",
            f"- **PLINK flag:** "
            f"`{step['plink_flag']}`",
            f"- **Example setting:** "
            f"`{step['example_setting']}`",
            f"- **Note:** "
            f"{step['note']}",
            f"- **Source:** "
            f"{step['source']}",
            ""
        ])

    return "\n".join(
        lines
    )


def format_commands(
    command_plan
):
    lines = [
        "## PLINK Commands",
        ""
    ]

    for item in command_plan:
        lines.append(
            f"### {item['step']}. "
            f"{item['step_name']}"
        )

        lines.append(
            f"{item['purpose']}"
        )

        for cmd in item[
            "commands"
        ]:
            lines.extend([
                "```bash",
                cmd,
                "```"
            ])

        lines.append(
            f"Source: "
            f"{item['source']}"
        )

        lines.append("")

    return "\n".join(
        lines
    )


def format_validation(
    title,
    errors,
    revisions
):
    if errors:
        status = "❌ FAIL"

        detail = "\n".join(
            f"- {e}"
            for e in errors
        )
    else:
        status = "✅ PASS"
        detail = (
            "- No remaining "
            "validator errors."
        )

    return (
        f"## {title}\n\n"
        f"**Status:** {status}\n\n"
        f"**Revision count:** "
        f"{revisions}\n\n"
        f"{detail}"
    )

## 17. Final Report Builder

In [ ]:
def build_final_report(
    study_info,
    qc_plan,
    command_plan,
    plan_errors,
    command_errors,
    plan_iterations,
    command_iterations
):
    covariates = (
        ", ".join(
            study_info.get(
                "covariates",
                []
            )
        )
        or "None"
    )

    workflow = " → ".join(
        step["name"]
        for step in qc_plan
    )

    sources = sorted({
        sid.strip()
        for step in qc_plan
        for sid in str(
            step.get(
                "source",
                ""
            )
        ).split(",")
        if sid.strip()
    })

    lines = [
        "# PLINK / GWAS QC Workflow Agent — Final Report",
        "",
        "## Study Summary",
        "",
        f"- **Study design:** "
        f"{study_info['study_design']}",
        f"- **Data format:** "
        f"{study_info['data_format']}",
        f"- **Sample size:** "
        f"{study_info['sample_size']}",
        f"- **Phenotype:** "
        f"{study_info['phenotype']}",
        f"- **Covariates:** "
        f"{covariates}",
        f"- **PCA:** "
        f"{study_info['need_pca']}",
        f"- **Relatedness:** "
        f"{study_info['need_relatedness']}",
        f"- **Association:** "
        f"{study_info['association']}",
        "",
        "## Recommended Workflow",
        "",
        workflow,
        "",
        "## Validation Summary",
        "",
        f"- Plan Check: "
        f"{'PASS' if not plan_errors else 'FAIL'}",
        f"- Plan revisions: "
        f"{plan_iterations}",
        f"- Command Check: "
        f"{'PASS' if not command_errors else 'FAIL'}",
        f"- Command revisions: "
        f"{command_iterations}",
        "",
        "## Knowledge Sources",
        "",
        ", ".join(
            sources
        ),
        "",
        "## Important Notes",
        "",
        "- QC threshold values shown by this system are examples, not universal standards.",
        "- LD-pruned variants may be used for PCA / relatedness, but the final GWAS should not automatically be restricted to only the pruned SNP subset.",
        "- Generated commands should still be reviewed in the context of the actual study design before real analysis."
    ]

    return "\n".join(
        lines
    )

# Part E — End-to-End Agent

## 18. Run Agent Function

In [ ]:
def run_agent(
    study_design_label,
    data_format_label,
    sample_size,
    phenotype,
    covariates_text,
    need_pca,
    need_relatedness,
    association_label,
    progress=gr.Progress()
):
    try:
        progress(0.03, desc="Preparing study information...")

        covariates = [
            x.strip()
            for x in (covariates_text or "").split(",")
            if x.strip()
        ]

        study_info = {
            "study_design": "case-control",
            "data_format": "plink_binary",
            "sample_size": int(sample_size),
            "phenotype": phenotype.strip() or "Disease Status",
            "covariates": covariates,
            "need_pca": bool(need_pca),
            "need_relatedness": bool(need_relatedness),
            "association": "logistic"
        }

        # ---------- Retrieval ----------
        progress(0.12, desc="Retrieving Wiki knowledge...")

        retrieved_docs, allowed_source_ids = retrieve_for_study(study_info)
        rag_context = build_rag_context(retrieved_docs)
        required_steps = build_required_steps(study_info)

        progress(0.25, desc="Wiki retrieval completed.")

        # ---------- Plan ----------
        progress(0.32, desc="Generating QC plan...")

        qc_plan = generate_qc_plan(
            study_info,
            required_steps,
            rag_context,
            allowed_source_ids
        )

        plan_iterations = 0

        progress(0.43, desc="Validating QC plan...")

        while True:
            plan_errors = validate_qc_plan(
                study_info,
                required_steps,
                qc_plan,
                allowed_source_ids
            )

            if not plan_errors:
                break

            if plan_iterations >= MAX_PLAN_ITERATIONS:
                break

            plan_iterations += 1

            progress(
                min(0.43 + 0.05 * plan_iterations, 0.58),
                desc=(
                    "QC plan failed validation. "
                    f"Revising plan ({plan_iterations}/{MAX_PLAN_ITERATIONS})..."
                )
            )

            qc_plan = revise_qc_plan(
                study_info,
                required_steps,
                rag_context,
                allowed_source_ids,
                qc_plan,
                plan_errors
            )

        if plan_errors:
            progress(
                0.60,
                desc="QC plan still has errors. Human review required."
            )

            retrieved_md = format_retrieved_knowledge(retrieved_docs)
            plan_md = format_qc_plan(qc_plan)
            plan_check_md = format_validation(
                "Plan Check",
                plan_errors,
                plan_iterations
            )

            stop_message = (
                "# Workflow stopped\n\n"
                "QC Plan still contains validator errors after "
                "maximum revisions. Human review is required."
            )

            return (
                retrieved_md,
                plan_md,
                plan_check_md,
                "Command generation not run.",
                "Command validation not run.",
                stop_message
            )

        progress(0.60, desc="QC plan passed validation.")

        # ---------- Commands ----------
        progress(0.66, desc="Generating PLINK commands...")

        command_plan = generate_commands(
            study_info,
            qc_plan
        )

        command_iterations = 0

        progress(0.76, desc="Validating PLINK commands...")

        while True:
            command_errors = validate_command_plan(
                study_info,
                qc_plan,
                command_plan
            )

            if not command_errors:
                break

            if command_iterations >= MAX_COMMAND_ITERATIONS:
                break

            command_iterations += 1

            progress(
                min(0.76 + 0.05 * command_iterations, 0.90),
                desc=(
                    "Command validation failed. "
                    f"Revising commands ({command_iterations}/{MAX_COMMAND_ITERATIONS})..."
                )
            )

            command_plan = revise_commands(
                study_info,
                qc_plan,
                command_plan,
                command_errors
            )

        progress(0.91, desc="Command validation completed.")

        # ---------- Outputs ----------
        progress(0.95, desc="Building final report...")

        retrieved_md = format_retrieved_knowledge(retrieved_docs)
        plan_md = format_qc_plan(qc_plan)
        plan_check_md = format_validation(
            "Plan Check",
            plan_errors,
            plan_iterations
        )
        commands_md = format_commands(command_plan)
        command_check_md = format_validation(
            "Command Check",
            command_errors,
            command_iterations
        )
        final_report = build_final_report(
            study_info,
            qc_plan,
            command_plan,
            plan_errors,
            command_errors,
            plan_iterations,
            command_iterations
        )

        progress(1.0, desc="Workflow completed.")

        return (
            retrieved_md,
            plan_md,
            plan_check_md,
            commands_md,
            command_check_md,
            final_report
        )

    except RateLimitError as e:
        progress(
            1.0,
            desc="Groq rate limit reached."
        )

        rate_message = (
            "# Groq Rate Limit Reached\n\n"
            "The workflow logic did not fail. "
            "Groq rejected the request because the current API rate/token "
            "limit has been reached.\n\n"
            "**What to do:** wait for the retry/reset period shown by Groq, "
            "then run the workflow again.\n\n"
            "To avoid unnecessary API usage, this app does **not** retry a "
            "429 rate-limit error automatically.\n\n"
            f"Technical message:\n`{str(e)}`"
        )

        return (
            rate_message,
            rate_message,
            rate_message,
            rate_message,
            rate_message,
            rate_message
        )

    except Exception as e:
        progress(1.0, desc="Workflow failed.")

        error_message = (
            "# Runtime Error\n\n"
            f"`{type(e).__name__}: {str(e)}`"
        )

        return (
            error_message,
            error_message,
            error_message,
            error_message,
            error_message,
            error_message
        )


# Part F — Gradio UI

## 19. 建立介面

這一版 UI 先以「功能與流程清楚」為主。

目前支援範圍：
- Case-control GWAS
- PLINK BED/BIM/FAM
- Logistic regression

之後若需要，可以再擴充其他 study design。

In [ ]:
with gr.Blocks(
    title=(
        "PLINK / GWAS "
        "QC Workflow Agent"
    ),
    fill_width=True
) as demo:

    gr.Markdown(
        '''
# 🧬 PLINK / GWAS QC Workflow Agent

根據研究條件自動：

**查詢 Wiki → 規劃 QC → 驗證 → 產生 PLINK 指令 → 再驗證**

> Threshold 為教學 / 分析範例，不代表所有研究都應固定使用相同設定。
'''
    )

    with gr.Row():

        with gr.Column(
            scale=1
        ):
            gr.Markdown(
                "## Study Information"
            )

            study_design = (
                gr.Dropdown(
                    choices=[
                        "Case-control GWAS"
                    ],
                    value=(
                        "Case-control GWAS"
                    ),
                    label="Study Design"
                )
            )

            data_format = (
                gr.Dropdown(
                    choices=[
                        "PLINK BED/BIM/FAM"
                    ],
                    value=(
                        "PLINK BED/BIM/FAM"
                    ),
                    label="Data Format"
                )
            )

            sample_size = (
                gr.Number(
                    value=3000,
                    precision=0,
                    label="Sample Size"
                )
            )

            phenotype = (
                gr.Textbox(
                    value=(
                        "Disease Status"
                    ),
                    label="Phenotype"
                )
            )

            covariates = (
                gr.Textbox(
                    value="Age, Sex",
                    label=(
                        "Covariates "
                        "(comma separated)"
                    )
                )
            )

            need_pca = (
                gr.Checkbox(
                    value=True,
                    label="Need PCA"
                )
            )

            need_relatedness = (
                gr.Checkbox(
                    value=True,
                    label=(
                        "Need Relatedness "
                        "Check"
                    )
                )
            )

            association = (
                gr.Dropdown(
                    choices=[
                        "Logistic Regression"
                    ],
                    value=(
                        "Logistic Regression"
                    ),
                    label=(
                        "Association Method"
                    )
                )
            )

            run_button = (
                gr.Button(
                    "▶ Run Workflow",
                    variant="primary"
                )
            )

        with gr.Column(
            scale=2
        ):

            with gr.Tabs():

                with gr.Tab(
                    "Retrieved Knowledge"
                ):
                    retrieved_output = (
                        gr.Markdown()
                    )

                with gr.Tab(
                    "QC Plan"
                ):
                    plan_output = (
                        gr.Markdown()
                    )

                with gr.Tab(
                    "Plan Check"
                ):
                    plan_check_output = (
                        gr.Markdown()
                    )

                with gr.Tab(
                    "PLINK Commands"
                ):
                    command_output = (
                        gr.Markdown()
                    )

                with gr.Tab(
                    "Command Check"
                ):
                    command_check_output = (
                        gr.Markdown()
                    )

                with gr.Tab(
                    "Final Report"
                ):
                    final_output = (
                        gr.Markdown()
                    )

    run_button.click(
        fn=run_agent,
        inputs=[
            study_design,
            data_format,
            sample_size,
            phenotype,
            covariates,
            need_pca,
            need_relatedness,
            association
        ],
        outputs=[
            retrieved_output,
            plan_output,
            plan_check_output,
            command_output,
            command_check_output,
            final_output
        ],
        show_progress="full"
    )

demo.queue()

## 20. 啟動 Web App

In [ ]:
START_DEMO = True

if START_DEMO:
    demo.queue(
        max_size=8,
        default_concurrency_limit=1
    )

    demo.launch(
        share=True,
        inline=True,
        debug=False,
        show_error=True
    )
else:
    print(
        "尚未啟動；確認公開連結與 API 額度提醒後，"
        "將 START_DEMO 改成 True。"
    )


STOP_DEMO = False

if STOP_DEMO:
    demo.close()
    print("展示服務已停止。")


# 21. 最終測試方式

啟動後先用預設案例：

```text
Study Design: Case-control GWAS
Data Format: PLINK BED/BIM/FAM
Sample Size: 3000
Phenotype: Disease Status
Covariates: Age, Sex
Need PCA: Yes
Need Relatedness: Yes
Association: Logistic Regression
```

按：

```text
Run Workflow
```

應依序看到：

### Retrieved Knowledge
多個 Wiki chapters / Source IDs

### QC Plan
完整 QC 流程

### Plan Check
```text
PASS
```

### PLINK Commands
各步驟 command

### Command Check
```text
PASS
```

### Final Report
研究摘要、workflow、驗證結果與來源。

---

# 到這一步代表 MVP 完成

期末專案核心已經形成：

```text
RAG
→ Planning
→ Plan Validation
→ Loop
→ Command Generation
→ Command Validation
→ Loop
→ Final Report
→ Gradio UI
```

下一階段不再是增加主要功能，而是：

1. 實際測試 3 個案例
2. 保存 FAIL → PASS 紀錄
3. 把程式整理到 GitHub
4. 補 README / workflow graph
5. 準備期末報告或簡報

## v2：Workflow 進度顯示

按下 **Run Workflow** 後，會顯示目前執行階段，例如：

```text
Preparing study information...
Retrieving Wiki knowledge...
Generating QC plan...
Validating QC plan...
Generating PLINK commands...
Validating PLINK commands...
Building final report...
Workflow completed.
```

若 Validator 失敗並觸發 loop，也會顯示目前 revision 次數。


## v3：Conditional Workflow 修正

LD Pruning 不再被視為所有案例都固定必做的步驟。

目前規則：

```text
PCA=False, Relatedness=False → 不要求 LD Pruning
PCA=False, Relatedness=True  → LD Pruning → Relatedness
PCA=True,  Relatedness=False → LD Pruning → PCA
PCA=True,  Relatedness=True  → LD Pruning → Relatedness + PCA
```

Final association 仍使用 post-QC association dataset，不會自動只使用 LD-pruned SNP subset。


## v4：Robust JSON Output

為避免 Groq strict JSON Schema 偶發 `400 json_validate_failed`，最終版改為：

```text
Groq JSON Object Mode
→ Python JSON shape validation
→ 若格式錯誤，自動重試（最多 3 次）
→ Plan / Command semantic Validator
```

因此 API 只負責產生合法 JSON，真正的結構與分析邏輯由本專案自己的 Python Validator 控制。


## v5：Rate-limit-safe

若 Groq 回傳 `429 rate_limit_exceeded`：

```text
立即停止 LLM 重試
→ 保留 API 額度
→ UI 顯示 Groq Rate Limit Reached
→ 等待 retry/reset 後再執行
```

只有 JSON 格式 / shape 問題才會進行最多 3 次自動重試。
